# P04 · Notebook 19 · Array front ends built from the twins (stage 1 of the full program)

**Question:** in heavy factory noise (−6 dB), can a front end built from the MIMII "twins" beat delay-and-sum, the array of notebooks 11 and 13? It is tested for a known machine and for a new one with 10 clips, on all four machine types.

**The idea.** Every MIMII clip exists at 6, 0 and −6 dB, with the same machine sound and different noise. From the three levels the notebook separates, per frequency, the spatial covariance of the machine from that of the noise. No noise-only recording is needed. It then builds:
- **MVDR**, which points the array's nulls at the noise. It uses the real noise covariance and a measured transfer function. Notebook 12's MPDR had the machine inside its covariance and steered by delays only.
- **MWF**, the multichannel Wiener filter.
- **N2N**, a small neural post-filter trained by Noise2Noise with the quieter twin as its target, so no clean sound is needed.

**On the way it checks:**
- whether the twins share the same noise segment (if they do, the exact clean machine sound is available);
- a "twin SNR meter" that measures each front end's SNR from normal training clips;
- which noise level the DCASE 2020 benchmark used (a side check).

**Rules:** `results/mimii_beats/RULES_twin_front_ends_written_before_running.md`, written before this run, within `PROGRAM_full_written_before_running.md`.

**What it needs:**
- In Google Drive:
  - `P04_BEATs` (the BEATs checkpoint);
  - `P04_ArrayNoise` and `P04_ArrayNoise_Fan` (notebooks 11 and 13);
  - `P04_Encoders` (notebook 15).
- A GPU runtime (a T4 is enough) and about 60 GB of free disk.

**Time:**
- About 100 GB of downloads from Zenodo: 1–2 h, depending on Zenodo.
- Then about 2–2.5 h of work on a T4.
- Results are saved in Drive (`P04_TwinFrontEnds`) after each machine type. **If Colab disconnects, run all cells again:** finished parts are skipped.

**At the end** a zip (`p04_n19_results.zip`) downloads. Send it back here.

In [ ]:
# 1 · Setup
import os, sys, json, time, hashlib, struct, zipfile, subprocess, shutil, urllib.request, re, importlib, threading, math
from pathlib import Path
from collections import Counter, deque
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

TEST = os.environ.get("P04_TEST") == "1"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MY = Path("/content/drive/MyDrive")
    OUT, WORK, CKPT_DIR, ENC = MY / "P04_TwinFrontEnds", Path("/content/work"), MY / "P04_BEATs", MY / "P04_Encoders"
    ARR = {"pump": MY / "P04_ArrayNoise", "fan": MY / "P04_ArrayNoise_Fan"}
else:
    OUT, WORK = Path(os.environ.get("P04_OUT", "P04_TwinFrontEnds")), Path(os.environ.get("P04_WORK", "work"))
    CKPT_DIR, ENC = Path(os.environ.get("P04_CKPT_DIR", str(OUT))), Path(os.environ.get("P04_ENC_DIR", "P04_Encoders"))
    ARR = {"pump": Path(os.environ.get("P04_ARR_PUMP", "P04_ArrayNoise")), "fan": Path(os.environ.get("P04_ARR_FAN", "P04_ArrayNoise_Fan"))}
for p in (OUT, WORK, CKPT_DIR):
    p.mkdir(parents=True, exist_ok=True)
for m, p in ARR.items():
    if not p.exists():
        raise SystemExit(f"missing {p}: notebook {11 if m == 'pump' else 13}'s files must be in Drive")
if not ENC.exists():
    raise SystemExit(f"missing {ENC}: notebook 15's files must be in Drive")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda"                              # float16 autocast for the BEATs encoder only on the GPU
GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else ""
T_START = time.time()
print("device:", DEVICE, GPU or "(no GPU: this would be very slow)", "· free disk:", round(shutil.disk_usage(WORK).free / 1e9), "GB")


def md5(path, chunk=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sha256(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def download(url, dest):
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    try:
        if url.startswith("file://"):
            shutil.copy(url[7:], part)
        elif shutil.which("wget"):
            subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", str(part), url], check=True)
        else:
            urllib.request.urlretrieve(url, part)
        part.rename(dest)
    finally:
        if part.exists() and not dest.exists():
            part.unlink()

In [ ]:
# 2 · The official BEATs code (the same files as notebook 1)
BEATS_RAW = "https://raw.githubusercontent.com/microsoft/unilm/master/beats/"
CODE_SHA256 = {"BEATs.py": "27f289db7c56ce26f2ceb50d3719854b91b2dec1c2830d8b1dd8de1bbee19eeb",
               "backbone.py": "31c0378379a7e0f1d1069f9da444fb86890fe1ea078959a2dcd39640cdcadbaa",
               "modules.py": "edeb6b6cd6a784da749f932c3e0783c0bce556fc768d0d23a4d53d4b819eb424"}
CODE_DIR = WORK / "beats"; CODE_DIR.mkdir(exist_ok=True)
code_sha = {}
for f, want in CODE_SHA256.items():
    p = CODE_DIR / f
    if not p.exists():
        download(BEATS_RAW + f, p)
    code_sha[f] = sha256(p)
    print(f, "same as notebook 1" if code_sha[f] == want else f"CHANGED upstream ({code_sha[f][:12]}…)")
try:
    import torchaudio.compliance.kaldi
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchaudio"], check=True)
sys.path.insert(0, str(CODE_DIR))
from BEATs import BEATs, BEATsConfig

In [ ]:
# 3 · The checkpoint notebook 1 used (from Drive › P04_BEATs); its SHA-256 must match
CKPT_NAME, CKPT_SHA_N1 = "BEATs_iter3_plus_AS2M.pt", "d43cbfad4d7b56381c061d7a24774f908d4d94c72961f6eb1d9090ff18cd8d34"
ckpt = CKPT_DIR / CKPT_NAME
if not ckpt.exists():
    raise SystemExit(f"Put {CKPT_NAME} in Google Drive › MyDrive › P04_BEATs (notebook 1 left it there), then run again.")
CKPT_SHA = sha256(ckpt)
print(CKPT_NAME, "· same file as notebook 1:", CKPT_SHA == CKPT_SHA_N1)
if not TEST:
    assert CKPT_SHA == CKPT_SHA_N1, "stop: not the checkpoint notebook 1 used"
try:
    chk = torch.load(ckpt, map_location="cpu", weights_only=True)
except Exception:
    chk = torch.load(ckpt, map_location="cpu", weights_only=False)
cfg = BEATsConfig(chk["cfg"])
model = BEATs(cfg); model.load_state_dict(chk["model"]); model.eval().to(DEVICE)
print("layers:", cfg.encoder_layers, "· fine-tuned:", cfg.finetuned_model)

In [ ]:
# 4 · Every number below was fixed in the rules before this notebook read any clip
RULES = "results/mimii_beats/RULES_twin_front_ends_written_before_running.md"
PROGRAM = "results/mimii_beats/PROGRAM_full_written_before_running.md"
Z = "https://zenodo.org/records/3384388/files/"
MD5 = {"pump|6dB": "a09ba6060c10fc09cd4c8770213b0b9f", "pump|0dB": "488748295c3f60b25de07b58fe75b049", "pump|-6dB": "d20b783a0ff9c93d58f452f98c37b112",
       "fan|6dB": "0890f7d3c2fd8448634e69ff1d66dd47", "fan|0dB": "6354d1cc2165c52168f9ef1bcd9c7c52", "fan|-6dB": "f02ae808a58d84b6815b7ec38ff30879",
       "valve|6dB": "fe5fb7c337cd701b1d31dc641e621892", "valve|0dB": "178478eb0d11c79080a35562bfdeee71", "valve|-6dB": "fdfaf185fea61b21e11952a070a4ada7",
       "slider|6dB": "838c2b3441858359c4704ef13a1b27ff", "slider|0dB": "4d674c21474f0646ecd75546db6c0c4e", "slider|-6dB": "49913eda7d37f182cbf8ed5c984140e0"}
ZIPS = {}
for key, h in MD5.items():
    t_, lv_ = key.split("|"); nm = f"{lv_.replace('dB', '_dB')}_{t_}.zip"
    ZIPS[key] = (Z + nm + "?download=1", nm, h)
DCASE = ("https://zenodo.org/records/3678171/files/dev_data_pump.zip?download=1", "dev_data_pump.zip", "90e7091ef722b7238a7f1009365779cd")
GB = {"pump": 23.8, "fan": 31.5, "valve": 22.4, "slider": 22.6}    # size of each type's three zips (record page), for the disk check
if TEST:
    _d = json.loads(os.environ["P04_TEST_DATA"]); DCASE = tuple(_d.pop("dcase")); ZIPS = {k: tuple(v) for k, v in _d.items()}
TYPES, MAIN, AUX = ["pump", "fan", "valve", "slider"], ["pump", "fan"], ["valve", "slider"]
LEVELS = ["6dB", "0dB", "-6dB"]
GAMMA = {"6dB": 6.0, "0dB": 0.0, "-6dB": -6.0}
KAPPA = {lv: 10 ** (-g / 10) for lv, g in GAMMA.items()}          # noise power relative to 0 dB (MIMII mixing recipe)
AMPL = {lv: 10 ** (-g / 20) for lv, g in GAMMA.items()}           # noise amplitude relative to 0 dB
SR, L, H = 16000, 1024, 512                                        # beamformer STFT
LOAD, EIG_FLOOR, TR_FLOOR = 1e-3, 1e-6, 1e-9
NFFT, HOP, UPS, MAXLAG = 1024, 512, 16, 16                         # GCC-PHAT exactly as notebook 11
TWIN_N, METER_N, CAL_N, G0E_N = 50, 100, 10, 16
TAU_TOL, G0C_MACHINE, G0C_AVG = 0.07, 0.02, 0.005
SHOTS = [1, 3, 10]
SPLITS = ["A", "B", "B+1", "B+3", "B+10"]
FOLDS = {1: ["id_00", "id_02"], 2: ["id_04", "id_06"]}
N2N_CFG = dict(n_fft=512, hop=256, hidden=256, layers=2, epochs=30, batch=32, lr=1e-3, lr_end=1e-5, clip=5.0, crop_s=4.0,
               val_every=20, norm_crops=200, in_levels=["-6dB", "0dB"], seed=0)
BEATS_BATCH = 16
PLAN = {("MAIN", "-6dB"): ["CH0", "DS", "MVDR", "MWF", "MVDRd", "MVDR10"], ("MAIN", "0dB"): ["CH0", "DS", "MVDR", "MWF"], ("MAIN", "6dB"): [],
        ("AUX", "-6dB"): ["CH0", "DS", "MVDR", "MWF"], ("AUX", "0dB"): ["CH0", "DS", "MVDR", "MWF"], ("AUX", "6dB"): ["CH0", "DS"]}
PLAN_ECHO = {"MAIN": ["MVDR", "MWF"], "AUX": ["CH0", "DS", "MVDR", "MWF"]}    # −6 dB; pumps' and fans' CH0 and DS come from notebook 15
METER_FE = ["CH0", "DS", "MVDR", "MWF", "MVDRd", "MVDR10"]
FE_ALL = ["CH0", "DS", "MVDR", "MWF", "N2N"]
CANDIDATES = ["MVDR", "MWF", "N2N"]
BAR_F1, BAR_F2 = 0.01, 0.01
REC = {"pump": {"CH0": 0.8234, "DS": 0.8795}, "fan": {"CH0": 0.7255, "DS": 0.8200}}    # notebooks 11 and 13, −6 dB, Split A (BF = DS)
REC_ECHO = {"pump": 0.8317, "fan": 0.8564}                                           # notebook 15, ECHO BF, −6 dB, Split A
MPDR = {"pump": {"A": 0.8713, "B+10": 0.8437}, "fan": {"A": 0.8228, "B+10": 0.6780}} # notebooks 12 and 13, BEATs, −6 dB
PUBLISHED_M6 = {"pump": 0.8977, "fan": 0.8778, "valve": 0.9794, "slider": 0.9441}  # arXiv 2309.07500, Table 1, −6 dB (indicative)
ECHO_HF, ECHO_SHA_N15 = "yucongzh/echo-small-0824", "1cc306d581c97604d70e451d24e8616b517f4dff5b4c488a7ea1bc860fbdafcc"
BEATS_FILE = {"pump": "array_{lv}_pump.npz", "fan": "fan_{lv}.npz"}                 # notebooks 11 and 13
if TEST:
    N2N_CFG.update(json.loads(os.environ.get("P04_TEST_CFG", "{}")).get("n2n", {}))
    G0E_N = 4
print("rules:", RULES)
print("noise power per level (relative to 0 dB):", {k: round(v, 5) for k, v in KAPPA.items()})

In [ ]:
# 5 · Downloads (checked against the MD5 on the Zenodo record), a background download for the next type, readers, notebook 11's splits
CHECKED = {}                                        # files whose MD5 was already checked in this session


def get_file(entry):
    url, name, want = entry
    dest = WORK / name
    if CHECKED.get(name) == want and dest.exists():
        return dest
    if dest.exists() and md5(dest) == want:
        CHECKED[name] = want
        return dest
    t = time.time(); download(url, dest)
    got = md5(dest)
    print(f"{name}: downloaded in {time.time() - t:.0f} s · MD5 matches Zenodo: {got == want}", flush=True)
    if got != want:
        dest.unlink()
        raise SystemExit(f"stop: {name} is not the published file")
    CHECKED[name] = want
    return dest


PREFETCH = {}


def prefetch(t):
    # download the next machine type while this one trains its denoisers (only if the disk has room)
    if t is None or t in PREFETCH or (OUT / f"done_{t}.json").exists():
        return
    if shutil.disk_usage(WORK).free / 1e9 < GB.get(t, 0) * 1.05 + 8:
        print(f"  (not enough free disk to download {t} in the background; it will download when its turn comes)")
        return

    def job():
        for lv in LEVELS:
            try:
                get_file(ZIPS[f"{t}|{lv}"])
            except BaseException as e:
                print(f"  (background download of {t} {lv} failed: {repr(e)[:150]}; it will be retried)", flush=True)
    th = threading.Thread(target=job, daemon=True); th.start(); PREFETCH[t] = th
    print(f"  background download of {t} started")


def read_all(data):
    # WAV bytes (plain or WAVE_FORMAT_EXTENSIBLE, 16-bit PCM) -> (samples, channels) float32 in [-1, 1], sample rate
    assert data[:4] == b"RIFF" and data[8:12] == b"WAVE", "not a WAV file"
    pos, fmt, raw = 12, None, None
    while pos + 8 <= len(data):
        cid, size = data[pos:pos + 4], struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if cid == b"fmt ":
            tag, ch, sr, _, _, bits = struct.unpack("<HHIIHH", body[:16])
            if tag == 0xFFFE:
                tag = struct.unpack("<H", body[24:26])[0]
            fmt = (tag, ch, sr, bits)
        elif cid == b"data":
            raw = body
        pos += 8 + size + (size & 1)
    tag, ch, sr, bits = fmt
    assert tag == 1 and bits == 16, f"expected 16-bit PCM, got format {tag}, {bits}-bit"
    nf = len(raw) // (2 * ch)
    return np.frombuffer(raw[:nf * 2 * ch], dtype="<i2").reshape(-1, ch).astype(np.float32) / 32768.0, sr


def parse(names):
    pump = np.array([re.search(r"(id_\d+)", n).group(1) for n in names])
    label = np.array([1 if n.replace("\\", "/").split("/")[-2] == "abnormal" else 0 for n in names])
    return pump, label


def split_a(pump, label, fname, p):
    # the same split as notebooks 1 and 3: test = first N normal by name + all abnormal of pump p; train = the other normal
    idx = np.where(pump == p)[0]
    nor = idx[label[idx] == 0]; abn = idx[label[idx] == 1]
    nor = nor[np.argsort(fname[nor])]
    return nor[len(abn):], np.r_[nor[:len(abn)], abn]


def open_type(t):
    return {lv: zipfile.ZipFile(get_file(ZIPS[f"{t}|{lv}"])) for lv in LEVELS}


def read3(zfs, name):
    # one clip at the three levels (its twins): level → (samples, 8) float32
    out = {}
    for lv, zf in zfs.items():
        x, sr = read_all(zf.read(name))
        assert sr == SR and x.ndim == 2 and x.shape[1] == 8, (lv, name, sr, x.shape)
        if not TEST:
            assert x.shape[0] == 10 * SR, (lv, name, x.shape)
        out[lv] = x
    return out


def stream(fn, items, depth=8, workers=2):
    # fn(item) runs in background threads, results come back in order: unzipping overlaps the main thread's work
    with ThreadPoolExecutor(workers) as ex:
        q, it = deque(), iter(items)
        for item in it:
            q.append(ex.submit(fn, item))
            if len(q) >= depth:
                break
        while q:
            res = q.popleft().result()
            for item in it:
                q.append(ex.submit(fn, item))
                break
            yield res

In [ ]:
# 6 · Physics: notebook 11's delays and delay-and-sum, the STFT, the twin estimators, MVDR and MWF, the twin meter, and a smoke test
WIN = np.hanning(NFFT).astype(np.float32)


def cross_spectra(x):
    # x: (samples, 8). Sum over frames of conj(X_0)·X_m for m = 0..7 → (8, NFFT/2+1) complex
    n = (len(x) - NFFT) // HOP + 1
    idx = np.arange(NFFT)[None, :] + HOP * np.arange(n)[:, None]
    F = np.fft.rfft(x[idx] * WIN[None, :, None], axis=1)           # (frames, bins, 8)
    return np.einsum("fb,fbm->mb", np.conj(F[:, :, 0]), F)


def delays_from(G):
    # GCC-PHAT on summed cross-spectra G (8, bins) → delay of each channel against channel 0, in samples
    out = np.zeros(G.shape[0])
    for m in range(1, G.shape[0]):
        g = G[m] / np.maximum(np.abs(G[m]), 1e-12)
        r = np.fft.irfft(g, n=NFFT * UPS)
        K = MAXLAG * UPS
        rc = np.concatenate([r[-K:], r[:K + 1]])                     # lags −K..+K (in 1/UPS samples)
        out[m] = (np.argmax(rc) - K) / UPS
    return out


def delay_and_sum(X, tau, N):
    k = np.arange(X.shape[0])[:, None]
    return np.fft.irfft((X * np.exp(2j * np.pi * k * tau[None, :] / N)).mean(1), n=N).astype(np.float32)


def front_ends(x, tau, tau_same=None):
    # x: (samples, 8) → mono signals; BF advances channel m by tau[m] samples, then averages
    N = len(x)
    X = np.fft.rfft(x, axis=0)
    out = {"CH0": x[:, 0].copy(), "AVG": x.mean(1).astype(np.float32), "BF": delay_and_sum(X, tau, N)}
    if tau_same is not None:
        out["BFs"] = delay_and_sum(X, tau_same, N)
    return out



SQH = torch.sqrt(torch.hann_window(L, periodic=True, dtype=torch.float32)).to(DEVICE)


def stft(x):
    # (channels, samples) float32 tensor → (channels, bins, frames) complex64; centred, zero padding, square-root Hann
    return torch.stft(x, n_fft=L, hop_length=H, window=SQH, center=True, pad_mode="constant", return_complex=True)


def istft(Y, n):
    return torch.istft(Y, n_fft=L, hop_length=H, window=SQH, center=True, length=n)


def to_dev(x):
    # (samples, channels) numpy → (channels, samples) tensor on the device
    return torch.from_numpy(np.ascontiguousarray(x.T)).to(DEVICE)


def gram(X, Y=None):
    # sum over frames of x yᴴ in every bin → (bins, 8, 8) complex128
    Y = X if Y is None else Y
    return torch.einsum("mkt,nkt->kmn", X, Y.conj()).to(torch.complex128)


def herm(A):
    return 0.5 * (A + np.conj(np.swapaxes(A, -1, -2)))


def eig_clip(A, rel):
    # Hermitian part; eigenvalues floored at rel × the largest (rel = 0: negative eigenvalues set to 0)
    w, V = np.linalg.eigh(herm(A))
    w = np.maximum(w, rel * w[..., -1:]) if rel > 0 else np.maximum(w, 0.0)
    return (V * w[..., None, :]) @ np.conj(np.swapaxes(V, -1, -2))


def estimate(D):
    # D: level → mean covariance (bins, 8, 8). D_l = R_s + κ_l R. Returns R (noise at 0 dB), R_s (machine), median model error at 0 dB
    R = eig_clip((D["-6dB"] - D["6dB"]) / (KAPPA["-6dB"] - KAPPA["6dB"]), EIG_FLOOR)
    Rs = eig_clip(D["6dB"] - KAPPA["6dB"] * R, 0.0)
    err = np.linalg.norm(D["0dB"] - (Rs + KAPPA["0dB"] * R), axis=(1, 2)) / np.maximum(np.linalg.norm(D["0dB"], axis=(1, 2)), 1e-30)
    return R, Rs, float(np.median(err[1:]))


def loaded(A):
    return A + (LOAD * np.real(np.trace(A, axis1=1, axis2=2)) / A.shape[1])[:, None, None] * np.eye(A.shape[1])[None]


def w_mvdr(R, Rs):
    # Souden et al. 2010 (β = 0): w = R⁻¹R_s e₀ / tr(R⁻¹R_s); output = the machine at microphone 0 for a rank-one R_s
    Q = np.linalg.solve(loaded(R), Rs)
    tr = np.real(np.trace(Q, axis1=1, axis2=2))
    w = Q[:, :, 0] / np.where(tr > 0, tr, 1.0)[:, None]
    bad = tr <= TR_FLOOR * tr.max()
    w[bad] = 0.0
    w[bad, 0] = 1.0
    return w


def w_mwf(R, Rs, kappa):
    # multichannel Wiener filter for the machine at microphone 0, at the noise level κ
    return np.linalg.solve(loaded(Rs + kappa * R), Rs[:, :, :1])[:, :, 0]


def steering(tau):
    k = np.arange(L // 2 + 1)[:, None]
    return np.exp(-2j * np.pi * k * tau[None, :] / L)              # notebook 12's: x_m(t) = s(t − τ_m)


def w_mvdrd(R, tau):
    # diagnostic: MVDR with the twin noise covariance but notebook 12's delay steering
    d = steering(tau)
    Rid = np.linalg.solve(loaded(R), d[:, :, None])[:, :, 0]
    return Rid / np.einsum("km,km->k", np.conj(d), Rid)[:, None]


def beam(w, X):
    # w: (bins, 8) complex64 tensor, X: (8, bins, frames) → wᴴx: (bins, frames)
    return torch.einsum("km,mkt->kt", w.conj(), X)


def twin_stats(x):
    # microphone 0: c_tw = corr(x₋₆, x₆), c_d = corr(x₋₆ − x₆, x₀ − x₆)
    a, b, c = (x[lv][:, 0].astype(np.float64) for lv in ("-6dB", "6dB", "0dB"))

    def corr(u, v):
        u = u - u.mean(); v = v - v.mean()
        return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-30))
    return corr(a, b), corr(a - b, c - b)


def twin_case(ctw, cd):
    if cd >= 0.9:
        return "S"
    if cd <= 0.3 and ctw >= 0.2:
        return "I"
    return "unclear"


def separate(x6, xm6):
    # case S only: the exact machine sound from the 6 dB and −6 dB twins
    return (AMPL["-6dB"] * x6 - AMPL["6dB"] * xm6) / (AMPL["-6dB"] - AMPL["6dB"])


def meter_add(acc, key, Y, Z, case):
    # twin meter, per bin. I: Y = F(x_l), Z = F(x₆): signal = Re Σ Y Z*, noise = Σ|Y|² − signal. S: Z = F(s), noise = Σ|Y − Z|²
    if case == "I":
        ps = (Y * Z.conj()).real.sum(-1).double(); pn = (Y.abs() ** 2).sum(-1).double() - ps
    else:
        ps = (Z.abs() ** 2).sum(-1).double(); pn = ((Y - Z).abs() ** 2).sum(-1).double()
    a = acc.setdefault(key, [0.0, 0.0])
    a[0] = a[0] + ps; a[1] = a[1] + pn


def snr_db(ps, pn):
    ps, pn = np.asarray(ps)[1:], np.asarray(pn)[1:]                  # bins 1–512
    return float(10 * np.log10(max(ps.sum(), 1e-30) / max(pn.sum(), 1e-30)))


# smoke test: a machine with a short echo pattern, a loud coherent interferer, diffuse noise; three levels with independent noise
rng = np.random.default_rng(0)
Nn, NCL = 32000, 24


def fir(delays, rng):
    h = np.zeros((8, 48))
    for c, d in enumerate(delays):
        h[c, d] = 1.0; h[c, d + 11] = 0.3 * rng.standard_normal(); h[c, d + 23] = 0.15 * rng.standard_normal()
    return h


def through(sig, h):
    return np.stack([np.convolve(sig, h[c])[:len(sig)] for c in range(8)], 1)


h_m, h_i = fir([0, 3, 5, 2, 7, 4, 6, 1], rng), fir([6, 1, 0, 5, 2, 7, 3, 4], rng)
tt = np.arange(Nn) / SR
mach = [through(sum(np.sin(2 * np.pi * (90 + 7 * j) * k * tt + rng.uniform(0, 6)) / k for k in range(1, 15)) + 0.3 * rng.standard_normal(Nn), h_m)
        for j in range(NCL)]
a_c = np.mean([np.mean(m ** 2, 0) for m in mach], 0)


def noise_clip(rng):
    n = through(rng.standard_normal(Nn), h_i) + 0.25 * rng.standard_normal((Nn, 8))
    return n / np.sqrt(np.mean(n ** 2, 0))


twins = []
for j in range(NCL):
    twins.append({lv: (mach[j] + np.sqrt(a_c * KAPPA[lv]) * noise_clip(rng)).astype(np.float32) for lv in LEVELS})
Ds, D_true, fr = {lv: 0 for lv in LEVELS}, 0, 0
for j in range(NCL):
    X = {lv: stft(to_dev(twins[j][lv])) for lv in LEVELS}
    for lv in LEVELS:
        Ds[lv] = Ds[lv] + gram(X[lv])
    D_true = D_true + gram(stft(to_dev(mach[j].astype(np.float32))))
    fr += X["6dB"].shape[-1]
Rr, Rsr, err0 = estimate({lv: (Ds[lv] / fr).cpu().numpy() for lv in LEVELS})
Dt = (D_true / fr).cpu().numpy()
rel = float(np.median(np.linalg.norm(Rsr - Dt, axis=(1, 2))[1:] / np.linalg.norm(Dt, axis=(1, 2))[1:]))
tau_s = np.array([0, 3, 5, 2, 7, 4, 6, 1], dtype=float)            # the machine's true direct-path delays (DS at its best)
wv = torch.from_numpy(w_mvdr(Rr, Rsr).astype(np.complex64)).to(DEVICE)
wm = torch.from_numpy(w_mwf(Rr, Rsr, KAPPA["-6dB"]).astype(np.complex64)).to(DEVICE)


def out_snr(f):
    # true output SNR of a linear front end f: its output on the machine part against its output on the noise part
    s_out, n_out = [], []
    for j in range(NCL // 2, NCL):
        s_out.append(f(mach[j].astype(np.float32))); n_out.append(f(twins[j]["-6dB"] - mach[j].astype(np.float32)))
    return 10 * np.log10(sum(np.sum(a ** 2) for a in s_out) / sum(np.sum(b ** 2) for b in n_out))


snr = {"CH0": out_snr(lambda x: x[:, 0]),
       "DS": out_snr(lambda x: delay_and_sum(np.fft.rfft(x, axis=0), tau_s, len(x))),
       "MVDR": out_snr(lambda x: istft(beam(wv, stft(to_dev(x))), len(x)).cpu().numpy()),
       "MWF": out_snr(lambda x: istft(beam(wm, stft(to_dev(x))), len(x)).cpu().numpy())}
acc_t = {}
for j in range(NCL // 2):
    X6, Xm = stft(to_dev(twins[j]["6dB"])), stft(to_dev(twins[j]["-6dB"]))
    meter_add(acc_t, "CH0", Xm[0], X6[0], "I"); meter_add(acc_t, "MVDR", beam(wv, Xm), beam(wv, X6), "I")
m_ch0, m_mv = (snr_db(acc_t[k][0].cpu().numpy(), acc_t[k][1].cpu().numpy()) for k in ("CH0", "MVDR"))
pr = float(np.abs(istft(stft(to_dev(twins[0]["6dB"])), Nn).cpu().numpy().T - twins[0]["6dB"]).max())
xS = {lv: (mach[0] + AMPL[lv] * np.sqrt(a_c) * noise_clip(np.random.default_rng(5))).astype(np.float32) for lv in LEVELS}
sep_err = float(np.abs(separate(xS["6dB"], xS["-6dB"]) - mach[0]).max() / np.abs(mach[0]).max())
print(f"STFT reconstruction error {pr:.1e} · R_s recovered within {rel:.1%} (median over bins) · model error at 0 dB {err0:.3f}")
print("true output SNR at −6 dB (dB):", {k: round(v, 1) for k, v in snr.items()}, f"· twin meter: CH0 {m_ch0:.1f}, MVDR {m_mv:.1f}")
print(f"twin structure: independent {twin_case(*twin_stats(twins[0]))} · same segment {twin_case(*twin_stats(xS))} · exact separation error {sep_err:.1e}")
assert pr < 1e-4, "smoke test: the STFT does not reconstruct"
assert rel < 0.15, "smoke test: the twin estimator does not recover the machine covariance"
assert snr["MVDR"] > snr["DS"] + 3, "smoke test: MVDR does not beat delay-and-sum against a coherent interferer"
assert abs(m_ch0 - snr["CH0"]) < 1.5 and abs(m_mv - snr["MVDR"]) < 1.5, "smoke test: the twin meter does not measure the SNR"
assert twin_case(*twin_stats(twins[0])) == "I" and twin_case(*twin_stats(xS)) == "S" and sep_err < 1e-4, "smoke test: twin structure"
del Ds, D_true

In [ ]:
# 7 · Encoders: BEATs (float32 filter banks, float16 encoder on the GPU) and ECHO-small (notebook 15's loader), checked on a probe
@torch.no_grad()
def embed_beats(batch):
    x = batch.to(DEVICE).float() if torch.is_tensor(batch) else torch.from_numpy(np.stack(batch)).to(DEVICE)
    with torch.autocast(device_type=DEVICE, enabled=False):
        fb = model.preprocess(x.float())                                  # filter banks always in float32
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=AMP):
        f = model.patch_embedding(fb.unsqueeze(1))
        f = f.reshape(f.shape[0], f.shape[1], -1).transpose(1, 2)
        f = model.layer_norm(f)
        if model.post_extract_proj is not None:
            f = model.post_extract_proj(f)
        h, _ = model.encoder(model.dropout_input(f), padding_mask=None)
    return h.float().mean(1).cpu().numpy()


INFO = {}
def load_echo():
    # as on the ECHO model card: its own code and model.safetensors from the same Hugging Face snapshot
    from huggingface_hub import snapshot_download
    from safetensors.torch import load_file
    snap = Path(snapshot_download(ECHO_HF))
    INFO["ECHO"] = {"repo": ECHO_HF, "revision": snap.name, "weights_sha256": sha256(snap / "model.safetensors"),
                    "code_sha256": sha256(snap / "audioMAE_band_upgrade.py")}
    saved = {k: sys.modules.pop(k) for k in ("modules", "utils") if k in sys.modules}
    sys.path.insert(0, str(snap))
    try:
        mod = importlib.import_module("audioMAE_band_upgrade")
    finally:
        sys.path.remove(str(snap))
        for k in ("modules", "utils"):
            sys.modules.pop(k, None)
        sys.modules.update(saved)
    m = mod.AudioMAEWithBand(spec_len=2000, band_width=32, shift_size=16, in_chans=1, embed_dim=384, encoder_depth=12,
                             num_heads=6, mlp_ratio=4.0, freq_pos_emb_dim=384)
    missing, unexpected = m.load_state_dict(load_file(str(snap / "model.safetensors")), strict=False)
    assert not missing, f"stop: ECHO weights missing for {list(missing)[:5]}"
    INFO["ECHO"].update(loader="model card: its own code + model.safetensors", unexpected_keys=len(unexpected))
    return m.to(DEVICE).eval()




@torch.no_grad()
def emb_echo(m, waves):
    out = []
    for x in waves:
        u, _ = m.extract_features_from_audio(torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32))[None].to(DEVICE), sample_rate=SR)
        out.append(u.float().cpu().numpy())
    return np.stack(out)


class EchoStandIn(torch.nn.Module):
    # TEST ONLY: a fixed random projection of log band energies, shaped like ECHO's output. Never used on Colab.
    def __init__(self, seed=2, bands=7, dim=16):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.bands, self.P = bands, torch.randn(bands, 8, dim, generator=g)

    def extract_features_from_audio(self, x, sample_rate):
        S = torch.stft(x[0].float(), n_fft=400, hop_length=160, window=torch.hann_window(400, device=x.device), return_complex=True).abs().log1p().T
        e = torch.stack([b.mean(-1) for b in S.chunk(self.bands, dim=-1)], -1)
        q = torch.stack([e[i * e.shape[0] // 8:(i + 1) * e.shape[0] // 8].mean(0) for i in range(8)], 0)
        return torch.einsum("qk,kqd->kd", q, self.P.to(x.device)).flatten(), None


if TEST:
    ECHO = EchoStandIn().to(DEVICE).eval(); INFO["ECHO"] = {"repo": "stand-in (TEST)", "weights_sha256": "", "loader": "stand-in"}
else:
    ECHO = load_echo()
    if INFO["ECHO"]["weights_sha256"] != ECHO_SHA_N15:
        print("NOTE: the ECHO weights differ from notebook 15's; gate G0(e) decides whether its files still apply")
rng = np.random.default_rng(0)
probe = [(rng.standard_normal(10 * SR) * 0.1).astype(np.float32) for _ in range(3)]
eb = embed_beats(probe)
with torch.no_grad():
    ref32 = model.extract_features(torch.from_numpy(np.stack(probe)).to(DEVICE))[0].float().mean(1).cpu().numpy()
cos16 = float(np.min(np.sum(eb * ref32, 1) / (np.linalg.norm(eb, axis=1) * np.linalg.norm(ref32, axis=1))))
ee, ee1 = emb_echo(ECHO, probe), emb_echo(ECHO, probe[:1])
print(f"BEATs: {eb.shape[1]} numbers per clip · float16 encoder against float32: cosine ≥ {cos16:.5f}")
print(f"ECHO: {ee.shape[1]} numbers per clip · {INFO['ECHO']['loader']}")
assert np.isfinite(eb).all() and cos16 > 0.99, "stop: the float16 BEATs encoder departs from float32"
assert ee.ndim == 2 and np.isfinite(ee).all() and np.allclose(ee[:1], ee1, atol=1e-4), "stop: ECHO output"

In [ ]:
# 8 · The references in Drive (notebooks 11, 13, 15), the scorer, and gate G0(d)
def load_ref(m, lv):
    D = np.load(ARR[m] / BEATS_FILE[m].format(lv=lv), allow_pickle=False)
    return {"fname": D["fname"], "mid": D["pump"] if "pump" in D.files else D["mid"], "label": D["label"],
            "CH0": D["CH0_mean"].astype(np.float64), "DS": D["BF_mean"].astype(np.float64), "tau": json.loads(str(D["tau"]))}


REF = {(m, lv): load_ref(m, lv) for m in MAIN for lv in LEVELS}
TAU_DRIVE = {m: {p: np.array(v) for p, v in REF[(m, "6dB")]["tau"].items()} for m in MAIN}
REF_ECHO = {}
for m in MAIN:
    D = np.load(ENC / f"enc_{m}_-6dB.npz", allow_pickle=False)
    assert np.array_equal(D["fname"], REF[(m, "-6dB")]["fname"]), "stop: notebook 15's clips differ from notebook 11/13's"
    REF_ECHO[m] = {"CH0": D["ECHO_CH0"].astype(np.float64), "DS": D["ECHO_BF"].astype(np.float64)}


def unit(E):
    return E / np.linalg.norm(E, axis=1, keepdims=True)


def b1(R, X):
    return 1.0 - (unit(X) @ unit(R).T).max(1)


def b1_loo(R):
    S = unit(R) @ unit(R).T
    np.fill_diagonal(S, -np.inf)
    return 1.0 - S.max(1)


def fold_of(p):
    return 1 if p in FOLDS[1] else 2


def fixed(E):
    return lambda p, s: E


def folded(E1, E2):
    # Split A: the machine's own fold's denoiser; B and B+n: the other fold's, for every clip involved
    return lambda p, s: (E1 if fold_of(p) == 1 else E2) if s == "A" else (E2 if fold_of(p) == 1 else E1)


def evaluate(get, mid, label, names, fuse=None):
    # notebook 11's splits; get(p, split) → the embeddings used for machine p; fuse: second model, rescaled-sum fusion (notebook 15)
    ids = sorted(set(mid.tolist()))
    A = {p: split_a(mid, label, names, p) for p in ids}
    r = {s: {} for s in SPLITS}
    for p in ids:
        tr, te = A[p]; y = label[te]
        fleet = np.concatenate([A[q][0] for q in ids if q != p])
        for s, ref in (("A", tr), ("B", fleet), *[(f"B+{n}", np.r_[fleet, tr[:n]]) for n in SHOTS]):
            if fuse is None:
                E = get(p, s); sc = b1(E[ref], E[te])
            else:
                sc = 0.0
                for g in (get, fuse):
                    E = g(p, s); lo = b1_loo(E[ref]); sc = sc + (b1(E[ref], E[te]) - lo.mean()) / lo.std()
            r[s][p] = round(float(roc_auc_score(y, sc)), 4)
    for s in SPLITS:
        r[s]["avg"] = round(float(np.mean([r[s][p] for p in ids])), 4)
    return r


G0 = {"a": {}, "b": {}, "c": {}, "d": {}, "e": {}}
for m in MAIN:
    R6 = REF[(m, "-6dB")]
    for f in ("CH0", "DS"):
        G0["d"][f"BEATs|{m}|{f}"] = [evaluate(fixed(R6[f]), R6["mid"], R6["label"], R6["fname"])["A"]["avg"], REC[m][f]]
    G0["d"][f"ECHO|{m}|DS"] = [evaluate(fixed(REF_ECHO[m]["DS"]), R6["mid"], R6["label"], R6["fname"])["A"]["avg"], REC_ECHO[m]]
G0D_PASS = all(abs(a - b) < 5e-5 for a, b in G0["d"].values())
print("G0(d) · the Drive files at −6 dB, Split A (got, recorded):", G0["d"], "→", "pass" if G0D_PASS else "FAIL")
if not TEST and not G0D_PASS:
    raise SystemExit("stop: G0(d) failed, the Drive files do not give the recorded numbers")

In [ ]:
# 9 · Noise2Noise post-filter: the network, the training loop, inference, and a smoke test on synthetic twins
NF, NH = N2N_CFG["n_fft"], N2N_CFG["hop"]
NB = NF // 2 + 1
SQH2 = torch.sqrt(torch.hann_window(NF, periodic=True, dtype=torch.float32)).to(DEVICE)


def stft2(x):
    return torch.stft(x, n_fft=NF, hop_length=NH, window=SQH2, center=True, pad_mode="constant", return_complex=True)


def istft2(Y, n):
    return torch.istft(Y, n_fft=NF, hop_length=NH, window=SQH2, center=True, length=n)


class N2NNet(nn.Module):
    # log-power per bin → Linear → ReLU → 2-layer bidirectional GRU → Linear → sigmoid: a real mask per bin and frame
    def __init__(self):
        super().__init__()
        self.register_buffer("mu", torch.zeros(NB)); self.register_buffer("sd", torch.ones(NB))
        self.inp = nn.Linear(NB, N2N_CFG["hidden"])
        self.rnn = nn.GRU(N2N_CFG["hidden"], N2N_CFG["hidden"], num_layers=N2N_CFG["layers"], batch_first=True, bidirectional=True)
        self.out = nn.Linear(2 * N2N_CFG["hidden"], NB)

    def forward(self, Y):
        f = torch.log10(Y.real ** 2 + Y.imag ** 2 + 1e-10).transpose(1, 2)
        z, _ = self.rnn(torch.relu(self.inp((f - self.mu) / self.sd)))
        return torch.sigmoid(self.out(z)).transpose(1, 2)


def n2n_loss(M, Y, T):
    # per sample Σ|M·Y − T|² / Σ|Y|² (the weight depends on the input only, so the Noise2Noise optimum is unchanged)
    num = ((M * Y - T).abs() ** 2).sum((1, 2)); den = (Y.abs() ** 2).sum((1, 2)).clamp_min(1e-12)
    return (num / den).mean()


def val_loss(net, vx, vt):
    net.eval(); tot = idl = 0.0
    with torch.no_grad():
        for i in range(0, len(vx), 32):
            Y, T = stft2(vx[i:i + 32]), stft2(vt[i:i + 32]); M = net(Y)
            tot += float(n2n_loss(M, Y, T)) * len(Y); idl += float(n2n_loss(torch.ones_like(M), Y, T)) * len(Y)
    net.train()
    return tot / len(vx), idl / len(vx)


def fit(get, n_pairs, norm_x, vx, vt, seed, epochs=None):
    # get(index array, rng) → (inputs, targets) float32 tensors on the device; each epoch is one pass over a fresh permutation
    epochs = epochs or N2N_CFG["epochs"]
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    net = N2NNet().to(DEVICE)
    with torch.no_grad():
        Fx = torch.cat([torch.log10(stft2(c).abs() ** 2 + 1e-10).permute(0, 2, 1).reshape(-1, NB) for c in norm_x])
        net.mu.copy_(Fx.mean(0)); net.sd.copy_(Fx.std(0).clamp_min(1e-3))
    B = N2N_CFG["batch"]; steps = max(1, math.ceil(n_pairs / B))
    opt = torch.optim.Adam(net.parameters(), lr=N2N_CFG["lr"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps * epochs, eta_min=N2N_CFG["lr_end"])
    v0, vi0 = val_loss(net, vx, vt)
    log = [{"epoch": 0, "train": None, "val": round(v0, 6), "val_identity": round(vi0, 6), "seconds": 0.0}]
    for ep in range(epochs):
        t0 = time.time(); order = rng.permutation(n_pairs); run = 0.0
        for b in range(steps):
            x, tg = get(order[b * B:(b + 1) * B], rng)
            Y, T = stft2(x), stft2(tg)
            loss = n2n_loss(net(Y), Y, T)
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), N2N_CFG["clip"]); opt.step(); sched.step()
            run += float(loss.detach())
        v, vi = val_loss(net, vx, vt)
        log.append({"epoch": ep + 1, "train": round(run / steps, 6), "val": round(v, 6), "val_identity": round(vi, 6), "seconds": round(time.time() - t0, 1)})
    return net, log


@torch.no_grad()
def denoise(net, x):
    # x: (batch, samples) float32 tensor → the post-filtered signal
    Y = stft2(x)
    return istft2(net(Y) * Y, x.shape[1])


# smoke test: harmonic sounds with independent noise in input and target; the denoiser must beat doing nothing on held-out clips
rng = np.random.default_rng(1)
ns, nc = 16000, 96
tt = torch.arange(ns, dtype=torch.float32) / SR
clean = torch.stack([sum(torch.sin(2 * math.pi * float(f0) * k * tt + float(rng.uniform(0, 6))) / k for k in range(1, 9))
                     for f0 in rng.uniform(80, 400, nc)]).to(DEVICE)
nz = lambda g: torch.from_numpy(rng.standard_normal((nc, ns)).astype(np.float32)).to(DEVICE) * g
xin, xtg = clean + nz(2.0), clean + nz(1.0)
net_s, log_s = fit(lambda idx, r: (xin[idx], xtg[idx]), 80, [xin[:16]], xin[80:], xtg[80:], seed=0, epochs=15 if not TEST else 6)
v0, v, vi = log_s[0]["val"], log_s[-1]["val"], log_s[-1]["val_identity"]
print(f"N2N smoke test: validation loss {v0:.3f} untrained → {v:.3f} after {len(log_s) - 1} epochs · {vi:.3f} for doing nothing")
assert v < vi and v < v0, "smoke test: the Noise2Noise denoiser does not learn"
del net_s

In [ ]:
# 10 · One machine type at a time: calibration from training clips, the twin meter, every clip through every front end, Noise2Noise
STORE = WORK / "stores"; STORE.mkdir(exist_ok=True)


def to_int16(v):
    return np.clip(np.round(v * 32768.0), -32768, 32767).astype(np.int16)


def lvkey(lv):
    return lv.replace("-", "m")


def calibrate(t, zfs, names, mid, label):
    # pass A: covariances at the three levels, the 6 dB GCC-PHAT spectra, the twin structure; all from training clips only
    cal, t0 = {}, time.time()
    for p in sorted(set(mid.tolist())):
        tr, _ = split_a(mid, label, names, p)
        D, D10, C, G, tw, fr, fr10 = {lv: 0 for lv in LEVELS}, {lv: 0 for lv in LEVELS}, 0, 0, [], 0, 0
        for j, x in enumerate(stream(lambda i: read3(zfs, names[i]), tr)):
            G = G + cross_spectra(x["6dB"])
            X = {lv: stft(to_dev(x[lv])) for lv in LEVELS}
            for lv in LEVELS:
                g = gram(X[lv]); D[lv] = D[lv] + g
                if j < CAL_N:
                    D10[lv] = D10[lv] + g
            C = C + gram(X["6dB"], X["-6dB"])
            fr += X["6dB"].shape[-1]
            if j < CAL_N:
                fr10 += X["6dB"].shape[-1]
            if j < TWIN_N:
                tw.append(twin_stats(x))
        R, Rs, err = estimate({lv: (D[lv] / fr).cpu().numpy() for lv in LEVELS})
        R10, Rs10, err10 = estimate({lv: (D10[lv] / fr10).cpu().numpy() for lv in LEVELS})
        Cn = (C / fr).cpu().numpy()
        rho = float(np.real(np.trace(Cn - Rs, axis1=1, axis2=2))[1:].sum() / np.real(np.trace(R, axis1=1, axis2=2))[1:].sum())
        tau_own = delays_from(G)
        tau = TAU_DRIVE[t][p] if t in MAIN else tau_own
        w = {"MVDR": w_mvdr(R, Rs), "MVDRd": w_mvdrd(R, tau), "MVDR10": w_mvdr(R10, Rs10)}
        w.update({f"MWF|{lv}": w_mwf(R, Rs, KAPPA[lv]) for lv in LEVELS})
        tw = np.array(tw)
        cal[p] = {"R": R, "Rs": Rs, "w": w, "tau": tau, "tau_own": tau_own, "model_err": err, "model_err10": err10, "rho": rho,
                  "c_tw": float(np.median(tw[:, 0])), "c_d": float(np.median(tw[:, 1])), "n_train": int(len(tr)),
                  "rank1_share": float(np.median((np.linalg.eigvalsh(Rs)[:, -1] / np.maximum(np.real(np.trace(Rs, axis1=1, axis2=2)), 1e-30))[1:]))}
        print(f"  {t} {p}: {len(tr)} training clips · model error {err:.3f} · ρ {rho:+.3f} · c_tw {cal[p]['c_tw']:.2f} · c_d {cal[p]['c_d']:.2f}"
              f" · machine rank-one share {cal[p]['rank1_share']:.2f} · {time.time() - t0:.0f} s", flush=True)
    return cal


def meter(t, zfs, names, mid, label, cal, case):
    # twin SNR meter on the first METER_N training clips of each machine, for CH0, DS, MVDR, MWF, MVDRd and MVDR10 at −6 and 0 dB
    res = {}
    for p in sorted(set(mid.tolist())):
        tr, _ = split_a(mid, label, names, p)
        w = {f: torch.from_numpy(cal[p]["w"][f].astype(np.complex64)).to(DEVICE) for f in ("MVDR", "MVDRd", "MVDR10")}
        wm = {lv: torch.from_numpy(cal[p]["w"][f"MWF|{lv}"].astype(np.complex64)).to(DEVICE) for lv in LEVELS}
        acc = {}
        for x in stream(lambda i: read3(zfs, names[i]), tr[:METER_N]):
            ref = x["6dB"] if case == "I" else separate(x["6dB"], x["-6dB"])
            Xr = stft(to_dev(ref))
            ds_r = stft(to_dev(delay_and_sum(np.fft.rfft(ref, axis=0), cal[p]["tau"], len(ref))[:, None]))[0]
            for lv in ("-6dB", "0dB"):
                X = stft(to_dev(x[lv]))
                meter_add(acc, ("CH0", lv), X[0], Xr[0], case)
                meter_add(acc, ("DS", lv), stft(to_dev(delay_and_sum(np.fft.rfft(x[lv], axis=0), cal[p]["tau"], len(x[lv]))[:, None]))[0], ds_r, case)
                for f in ("MVDR", "MVDRd", "MVDR10"):
                    meter_add(acc, (f, lv), beam(w[f], X), beam(w[f], Xr), case)
                meter_add(acc, ("MWF", lv), beam(wm[lv], X), beam(wm[lv], Xr), case)
        res[p] = {f"{f}|{lv}": {"ps": a[0].cpu().numpy(), "pn": a[1].cpu().numpy(), "snr": snr_db(a[0].cpu().numpy(), a[1].cpu().numpy())}
                  for (f, lv), a in acc.items()}
    return res


def save_stats(t, cal, met, summ):
    arrs = {}
    for p, c in cal.items():
        arrs[f"{p}__R"], arrs[f"{p}__Rs"], arrs[f"{p}__tau"], arrs[f"{p}__tau_own"] = c["R"], c["Rs"], c["tau"], c["tau_own"]
        for f, w in c["w"].items():
            arrs[f"{p}__w__{f}"] = w
    for p, d in (met or {}).items():
        for k, v in d.items():
            arrs[f"{p}__meter__{k}__ps"], arrs[f"{p}__meter__{k}__pn"] = v["ps"], v["pn"]
    np.savez_compressed(OUT / f"stats_{t}.npz", summary=json.dumps(summ), **arrs)


def load_stats(t):
    D = np.load(OUT / f"stats_{t}.npz", allow_pickle=False)
    summ = json.loads(str(D["summary"]))
    cal = {p: {"w": {}, **summ["machines"][p]} for p in summ["machines"]}
    for k in D.files:
        if k == "summary" or "__meter__" in k:
            continue
        parts = k.split("__")
        if parts[1] == "w":
            cal[parts[0]]["w"][parts[2]] = D[k]
        else:
            cal[parts[0]][parts[1]] = D[k]
    met = {}
    for k in D.files:
        if "__meter__" in k and k.endswith("__ps"):
            p, _, fk, _ = k.split("__")
            met.setdefault(p, {})[fk] = {"ps": D[k], "pn": D[k[:-4] + "__pn"], "snr": snr_db(D[k], D[k[:-4] + "__pn"])}
    return cal, met, summ


def pass_b(t, zfs, names, mid, label, cal, base):
    # every clip at every level through the planned front ends → BEATs and ECHO; the N2N base at every level goes to the local store
    grp = "MAIN" if t in MAIN else "AUX"
    n = len(names); ns = read3(zfs, names[0])["6dB"].shape[0]
    mm = {lv: np.lib.format.open_memmap(STORE / f"{t}_{lv}.npy", mode="w+", dtype=np.int16, shape=(n, ns)) for lv in LEVELS}
    W = {p: {f: torch.from_numpy(w.astype(np.complex64)).to(DEVICE) for f, w in c["w"].items()} for p, c in cal.items()}
    E, buf, g0e, seen, t0 = {}, {}, {}, Counter(), time.time()

    def push(key, y):
        buf.setdefault(key, []).append(y)
        if len(buf[key]) == BEATS_BATCH:
            E.setdefault(key, []).append(embed_beats(buf[key])); buf[key] = []

    for i, x in enumerate(stream(lambda i: read3(zfs, names[i]), range(n))):
        p = mid[i]
        for lv in LEVELS:
            need = set(PLAN[(grp, lv)]) | {base} | (set(PLAN_ECHO[grp]) | ({"DS"} if t in MAIN else set()) if lv == "-6dB" else set())
            X = stft(to_dev(x[lv]))
            out = {}
            if "CH0" in need:
                out["CH0"] = np.ascontiguousarray(x[lv][:, 0])
            if "DS" in need:
                out["DS"] = delay_and_sum(np.fft.rfft(x[lv], axis=0), cal[p]["tau"], ns)
            for f in ("MVDR", "MVDRd", "MVDR10"):
                if f in need:
                    out[f] = istft(beam(W[p][f], X), ns).cpu().numpy()
            if "MWF" in need:
                out["MWF"] = istft(beam(W[p][f"MWF|{lv}"], X), ns).cpu().numpy()
            mm[lv][i] = to_int16(out[base])
            for f in PLAN[(grp, lv)]:
                push(("BEATs", lv, f), out[f])
            if lv == "-6dB":
                for f in PLAN_ECHO[grp]:
                    E.setdefault(("ECHO", lv, f), []).append(emb_echo(ECHO, [out[f]]))
                if t in MAIN and seen[p] < G0E_N:
                    g0e.setdefault("rows", []).append(i); g0e.setdefault("emb", []).append(emb_echo(ECHO, [out["DS"]])[0])
        seen[p] += 1
        if i % 500 == 0:
            print(f"  {t}: {i}/{n} clips · {time.time() - t0:.0f} s", flush=True)
    for key, b in buf.items():
        if b:
            E.setdefault(key, []).append(embed_beats(b))
    for lv in LEVELS:
        mm[lv].flush()
    (STORE / f"{t}.json").write_text(json.dumps({"names": names.tolist(), "base": base, "samples": ns}))
    arrs = {f"{enc}_{lvkey(lv)}_{f}": np.concatenate(v).astype(np.float32) for (enc, lv, f), v in E.items()}
    if g0e:
        arrs["ECHO_G0_rows"], arrs["ECHO_G0_DS"] = np.array(g0e["rows"]), np.stack(g0e["emb"]).astype(np.float32)
    np.savez_compressed(OUT / f"emb_{t}.npz", fname=names, mid=mid, label=label, **arrs)
    print(f"{t}: every clip through the front ends · {time.time() - t0:.0f} s")


def ensure_store(t, names, mid, cal, base):
    # the local store is rebuilt from the zips only if Colab lost it after a disconnect
    info = STORE / f"{t}.json"
    if info.exists() and all((STORE / f"{t}_{lv}.npy").exists() for lv in LEVELS):
        return {lv: np.load(STORE / f"{t}_{lv}.npy", mmap_mode="r") for lv in LEVELS}
    print(f"  {t}: rebuilding the local store from the zips (lost after a disconnect)")
    zfs = open_type(t); ns = read3(zfs, names[0])["6dB"].shape[0]
    mm = {lv: np.lib.format.open_memmap(STORE / f"{t}_{lv}.npy", mode="w+", dtype=np.int16, shape=(len(names), ns)) for lv in LEVELS}
    W = {p: {f: torch.from_numpy(w.astype(np.complex64)).to(DEVICE) for f, w in c["w"].items()} for p, c in cal.items()}
    for i, x in enumerate(stream(lambda i: read3(zfs, names[i]), range(len(names)))):
        for lv in LEVELS:
            p = mid[i]
            y = delay_and_sum(np.fft.rfft(x[lv], axis=0), cal[p]["tau"], ns) if base == "DS" else istft(beam(W[p]["MVDR"], stft(to_dev(x[lv]))), ns).cpu().numpy()
            mm[lv][i] = to_int16(y)
    for lv in LEVELS:
        mm[lv].flush()
    info.write_text(json.dumps({"names": names.tolist(), "base": base, "samples": ns}))
    return {lv: np.load(STORE / f"{t}_{lv}.npy", mmap_mode="r") for lv in LEVELS}


def crops(mm, rows, starts, n):
    return torch.from_numpy(np.stack([mm[r, s:s + n] for r, s in zip(rows, starts)]).astype(np.float32) / 32768.0).to(DEVICE)


def target(st, rows, starts, n, case):
    if case == "S":
        return (AMPL["-6dB"] * crops(st["6dB"], rows, starts, n) - AMPL["6dB"] * crops(st["-6dB"], rows, starts, n)) / (AMPL["-6dB"] - AMPL["6dB"])
    return crops(st["6dB"], rows, starts, n)


def train_n2n(t, k, st, mid, label, names, case):
    path = OUT / f"n2n_{t}_f{k}.pt"
    if path.exists():
        return torch.load(path, map_location="cpu", weights_only=False)
    trn, va = [], []
    for p in FOLDS[k]:
        if p in set(mid.tolist()):
            tr, _ = split_a(mid, label, names, p)
            v = tr[::N2N_CFG["val_every"]]; va += v.tolist(); trn += [i for i in tr if i not in set(v.tolist())]
    trn, va = np.array(trn), np.array(va)
    n, ns = int(N2N_CFG["crop_s"] * SR), st["6dB"].shape[1]
    pairs = [(r, lv) for r in trn for lv in N2N_CFG["in_levels"]]
    rng = np.random.default_rng(N2N_CFG["seed"] + k)
    norm_rows = rng.choice(trn, N2N_CFG["norm_crops"]); norm_st = rng.integers(0, ns - n + 1, len(norm_rows))
    norm_lv = rng.choice(N2N_CFG["in_levels"], len(norm_rows))
    norm_x = [torch.cat([crops(st[lv], [r], [s], n) for r, s, lv in zip(norm_rows[i:i + 32], norm_st[i:i + 32], norm_lv[i:i + 32])])
              for i in range(0, len(norm_rows), 32)]

    def get(idx, r):
        sel = [pairs[j] for j in idx]; rows = np.array([a for a, _ in sel]); sts = r.integers(0, ns - n + 1, len(sel))
        x = torch.cat([crops(st[lv], [a], [s], n) for (a, lv), s in zip(sel, sts)])
        return x, target(st, rows, sts, n, case)

    z0 = np.zeros(len(va), dtype=int)
    vx, vt = crops(st["-6dB"], va, z0, n), target(st, va, z0, n, case)
    t0 = time.time()
    net, log = fit(get, len(pairs), norm_x, vx, vt, seed=N2N_CFG["seed"] + k)
    rec = {"state": {k_: v.cpu() for k_, v in net.state_dict().items()}, "log": log, "cfg": N2N_CFG, "case": case,
           "base": json.loads((STORE / f"{t}.json").read_text())["base"], "train_clips": int(len(trn)), "val_clips": int(len(va)),
           "seconds": round(time.time() - t0, 1)}
    torch.save(rec, path)
    last = log[-1]
    print(f"  {t} fold {k}: N2N trained on {len(trn)} clips × {len(N2N_CFG['in_levels'])} levels · validation {last['val']:.4f}"
          f" against {last['val_identity']:.4f} for doing nothing · {rec['seconds']:.0f} s", flush=True)
    return rec


def embed_n2n(t, st, recs):
    out = OUT / f"emb_n2n_{t}.npz"
    if out.exists():
        return
    nets = {}
    for k, rec in recs.items():
        net = N2NNet().to(DEVICE); net.load_state_dict(rec["state"]); net.eval(); nets[k] = net
    E, t0, n = {}, time.time(), st["-6dB"].shape[0]
    for i in range(0, n, BEATS_BATCH):
        x = torch.from_numpy(np.asarray(st["-6dB"][i:i + BEATS_BATCH]).astype(np.float32) / 32768.0).to(DEVICE)
        for k, net in nets.items():
            y = denoise(net, x)
            E.setdefault(f"BEATs_m6dB_N2N{k}", []).append(embed_beats(y))
            E.setdefault(f"ECHO_m6dB_N2N{k}", []).append(emb_echo(ECHO, list(y.cpu().numpy())))
    np.savez_compressed(out, **{k: np.concatenate(v).astype(np.float32) for k, v in E.items()})
    print(f"  {t}: N2N embeddings · {time.time() - t0:.0f} s")


def dcase_check(zfs, names, mid, label):
    # side check: which MIMII level (microphone 0) did DCASE 2020 Task 2 use for pumps? Correlation of the first 2 s
    zd = zipfile.ZipFile(get_file(DCASE))
    dn = sorted(n for n in zd.namelist() if n.lower().endswith(".wav"))
    groups = {"normal": [n for n in dn if "/train/" in n and "normal_id_00_" in n][:20], "anomaly": [n for n in dn if "/test/" in n and "anomaly_id_00_" in n][:10]}
    m2 = 2 * SR
    dc = {g: np.stack([read_all(zd.read(n))[0][:m2, 0] for n in v]).astype(np.float64) for g, v in groups.items()}
    idx = np.where(mid == "id_00")[0]
    M = {lv: [] for lv in LEVELS}
    for x in stream(lambda i: read3(zfs, names[i]), idx):
        for lv in LEVELS:
            M[lv].append(x[lv][:m2, 0].astype(np.float64))
    nz = lambda A: (A - A.mean(1, keepdims=True)) / (np.linalg.norm(A - A.mean(1, keepdims=True), axis=1, keepdims=True) + 1e-30)

    def lagged(a, m, q=SR // 4):
        # best normalised correlation of a's middle 1.5 s with m over shifts of ±0.25 s
        d = a[q:m2 - q] - a[q:m2 - q].mean(); nd = len(d)
        c = np.correlate(m, d, mode="valid")
        cs, cs2 = np.cumsum(np.r_[0.0, m]), np.cumsum(np.r_[0.0, m ** 2])
        s1, s2 = cs[nd:] - cs[:-nd], cs2[nd:] - cs2[:-nd]
        return float(np.max(c / (np.linalg.norm(d) * np.sqrt(np.maximum(s2 - s1 ** 2 / nd, 1e-30)))))

    res = {"clips": {g: len(v) for g, v in groups.items()}, "levels": {}}
    for lv in LEVELS:
        Mr = np.stack(M[lv]); Ml = nz(Mr); r = {}
        for g, A in dc.items():
            C = nz(A) @ Ml.T; best = C.argmax(1); mx = C.max(1)
            lag = [lagged(a, Mr[j]) for a, j in zip(A, best)]
            r[g] = {"median_max_corr": round(float(np.median(mx)), 4), "min_max_corr": round(float(mx.min()), 4), "median_lagged": round(float(np.median(lag)), 4)}
        res["levels"][lv] = r
    hits = [lv for lv, r in res["levels"].items() if all(r[g]["median_max_corr"] >= 0.99 for g in r)]
    res["identified"] = hits[0] if len(hits) == 1 else None
    (OUT / "dcase_check.json").write_text(json.dumps(res, indent=1))
    print("DCASE 2020 side check:", res)
    zd.close()
    if IN_COLAB:
        (WORK / DCASE[1]).unlink(missing_ok=True)
    return res


def process_type(t, nxt=None):
    done = OUT / f"done_{t}.json"
    if done.exists():
        print(t, "already done"); return json.loads(done.read_text())
    if t in PREFETCH:
        PREFETCH[t].join()
    t0 = time.time(); times = {}
    zf6 = zipfile.ZipFile(get_file(ZIPS[f"{t}|6dB"]))
    names = np.array(sorted(n for n in zf6.namelist() if n.lower().endswith(".wav"))); zf6.close()
    mid, label = parse(names)
    ids = sorted(set(mid.tolist()))
    assert all(p in FOLDS[1] + FOLDS[2] for p in ids), f"unexpected machine IDs {ids}"
    if t in MAIN:
        G0["a"][t] = all(np.array_equal(names, REF[(t, lv)]["fname"]) for lv in LEVELS)
        print(f"G0(a) · {t}: the clips are notebook {11 if t == 'pump' else 13}'s:", G0["a"][t])
        if not G0["a"][t] and not TEST:
            raise SystemExit("stop: G0(a) failed")
    print(f"{t}: {len(names)} clips · per machine {dict(Counter(mid.tolist()))} · abnormal {int(label.sum())}")
    zfs = None
    if (OUT / f"stats_{t}.npz").exists():
        cal, met, summ = load_stats(t)
    else:
        zfs = open_type(t); times["download_s"] = round(time.time() - t0, 1)
        cal = calibrate(t, zfs, names, mid, label); times["calibration_s"] = round(time.time() - t0 - times["download_s"], 1)
        c_tw = float(np.median([c["c_tw"] for c in cal.values()])); c_d = float(np.median([c["c_d"] for c in cal.values()]))
        case = twin_case(c_tw, c_d)
        print(f"G1 · {t}: twin structure c_tw {c_tw:.2f} · c_d {c_d:.2f} → {case}")
        met, valid, base = {}, False, "DS"
        if case in ("I", "S"):
            met = meter(t, zfs, names, mid, label, cal, case)
            s6 = float(np.median([met[p]["CH0|-6dB"]["snr"] for p in ids])); s0 = float(np.median([met[p]["CH0|0dB"]["snr"] for p in ids]))
            valid = abs(s6 + 6) <= 1.5 and abs(s0) <= 1.5
            med = {f: float(np.median([met[p][f"{f}|-6dB"]["snr"] for p in ids])) for f in METER_FE}
            base = ("MVDR" if med["MVDR"] > med["DS"] else "DS") if valid else "DS"
            print(f"G1b · {t}: CH0 meter {s6:.2f} dB at −6 dB, {s0:.2f} dB at 0 dB →", "valid" if valid else "INVALID",
                  "· median SNR at −6 dB:", {f: round(v, 2) for f, v in med.items()}, "· N2N base:", base)
        summ = {"case": case, "c_tw": c_tw, "c_d": c_d, "meter_valid": valid, "base": base,
                "machines": {p: {k: v for k, v in c.items() if k not in ("R", "Rs", "w", "tau", "tau_own")} for p, c in cal.items()}}
        save_stats(t, cal, met, summ)
    for p, c in cal.items():
        c["tau"], c["tau_own"] = np.asarray(c["tau"]), np.asarray(c["tau_own"])
    if t in MAIN:
        G0["b"][t] = round(float(max(np.max(np.abs(cal[p]["tau_own"] - TAU_DRIVE[t][p])) for p in ids)), 4)
        print(f"G0(b) · {t}: our delays against Drive's, largest difference {G0['b'][t]} samples (bar {TAU_TOL})")
        if G0["b"][t] > TAU_TOL and not TEST:
            raise SystemExit("stop: G0(b) failed")
    if not (OUT / f"emb_{t}.npz").exists():
        zfs = zfs or open_type(t); tb = time.time()
        pass_b(t, zfs, names, mid, label, cal, summ["base"]); times["front_ends_s"] = round(time.time() - tb, 1)
        if t == "pump" and not (OUT / "dcase_check.json").exists():
            try:
                dcase_check(zfs, names, mid, label)
            except Exception as e:
                (OUT / "dcase_check.json").write_text(json.dumps({"failed": repr(e)[:500]})); print("DCASE side check failed:", repr(e)[:300])
    if zfs is not None:
        for zf in zfs.values():
            zf.close()
        if IN_COLAB:
            for lv in LEVELS:
                (WORK / ZIPS[f"{t}|{lv}"][1]).unlink(missing_ok=True)
    De = np.load(OUT / f"emb_{t}.npz", allow_pickle=False)
    if t in MAIN:
        worst_m = worst_a = 0.0
        for lv in ("-6dB", "0dB"):
            for f in ("CH0", "DS"):
                ours = evaluate(fixed(De[f"BEATs_{lvkey(lv)}_{f}"].astype(np.float64)), mid, label, names)
                ref = evaluate(fixed(REF[(t, lv)][f]), mid, label, names)
                for s in SPLITS:
                    worst_a = max(worst_a, abs(ours[s]["avg"] - ref[s]["avg"]))
                    worst_m = max(worst_m, max(abs(ours[s][p] - ref[s][p]) for p in ids))
        G0["c"][t] = {"worst_machine": round(worst_m, 4), "worst_average": round(worst_a, 4)}
        rows = De["ECHO_G0_rows"]; a, b = De["ECHO_G0_DS"].astype(np.float64), REF_ECHO[t]["DS"][rows]
        G0["e"][t] = round(float(np.min(np.sum(a * b, 1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)))), 6)
        print(f"G0(c) · {t}: float16 against float32, largest |ΔAUC| per machine {worst_m:.4f} (bar {G0C_MACHINE}), per average {worst_a:.4f} (bar {G0C_AVG})")
        print(f"G0(e) · {t}: ECHO DS against notebook 15, smallest cosine {G0['e'][t]:.6f} (bar 0.999)")
        if not TEST and (worst_m > G0C_MACHINE or worst_a > G0C_AVG or G0["e"][t] < 0.999):
            raise SystemExit("stop: G0(c) or G0(e) failed")
    prefetch(nxt)
    n2n = {"status": "skipped: twin structure unclear"}
    if summ["case"] in ("I", "S"):
        st = ensure_store(t, names, mid, cal, summ["base"]); tn = time.time()
        recs = {k: train_n2n(t, k, st, mid, label, names, summ["case"]) for k in (1, 2)}
        embed_n2n(t, st, recs); times["n2n_s"] = round(time.time() - tn, 1)
        n2n = {"status": "done", "base": summ["base"], "G3": {str(k): bool(r["log"][-1]["val"] < r["log"][-1]["val_identity"]) for k, r in recs.items()},
               "logs": {str(k): r["log"] for k, r in recs.items()}, "train_clips": {str(k): r["train_clips"] for k, r in recs.items()}}
    res = {"case": summ["case"], "c_tw": summ["c_tw"], "c_d": summ["c_d"], "meter_valid": summ["meter_valid"], "base": summ["base"],
           "machines": summ["machines"], "n2n": n2n, "times": times, "minutes": round((time.time() - t0) / 60, 1),
           "G0": {g: G0[g].get(t) for g in G0}}
    done.write_text(json.dumps(res))
    if IN_COLAB:
        for lv in LEVELS:
            (STORE / f"{t}_{lv}.npy").unlink(missing_ok=True)
    print(f"{t}: done in {res['minutes']} min")
    return res

In [ ]:
# 11 · Run the four machine types. Safe to re-run after a disconnect: finished parts are skipped
SUMMARY = {}
for j, t in enumerate(TYPES):
    SUMMARY[t] = process_type(t, nxt=TYPES[j + 1] if j + 1 < len(TYPES) else None)
for t in MAIN:                                                  # gates of types finished in an earlier session
    for g in ("a", "b", "c", "e"):
        if t not in G0[g] and SUMMARY[t]["G0"].get(g) is not None:
            G0[g][t] = SUMMARY[t]["G0"][g]
print(f"all four types done · {(time.time() - T_START) / 60:.0f} min in this session")

In [ ]:
# 12 · Scores (B1), the reported tables, and decisions F1–F2 exactly as written in the rules
EMB, META = {}, {}
for t in TYPES:
    D = np.load(OUT / f"emb_{t}.npz", allow_pickle=False)
    META[t] = (D["fname"], D["mid"], D["label"])
    for k in D.files:
        if k.startswith(("BEATs_", "ECHO_")) and not k.startswith("ECHO_G0"):
            enc, lk, f = k.split("_", 2); EMB[(enc, t, lk.replace("m", "-"), f)] = D[k].astype(np.float64)
    if (OUT / f"emb_n2n_{t}.npz").exists():
        Dn = np.load(OUT / f"emb_n2n_{t}.npz", allow_pickle=False)
        for enc in ("BEATs", "ECHO"):
            EMB[(enc, t, "-6dB", "N2N")] = (Dn[f"{enc}_m6dB_N2N1"].astype(np.float64), Dn[f"{enc}_m6dB_N2N2"].astype(np.float64))
for m in MAIN:
    EMB[("BEATs", m, "6dB", "CH0")], EMB[("BEATs", m, "6dB", "DS")] = REF[(m, "6dB")]["CH0"], REF[(m, "6dB")]["DS"]
    EMB[("ECHO", m, "-6dB", "CH0")], EMB[("ECHO", m, "-6dB", "DS")] = REF_ECHO[m]["CH0"], REF_ECHO[m]["DS"]


def getter(key):
    E = EMB[key]
    return folded(*E) if isinstance(E, tuple) else fixed(E)


RES = {}
for key in EMB:
    enc, t, lv, f = key
    fn, mid, label = META[t]
    RES[key] = evaluate(getter(key), mid, label, fn)
FUSED = {}
for t in TYPES:
    fn, mid, label = META[t]
    for f in FE_ALL:
        if ("BEATs", t, "-6dB", f) in EMB and ("ECHO", t, "-6dB", f) in EMB:
            FUSED[(t, f)] = evaluate(getter(("BEATs", t, "-6dB", f)), mid, label, fn, fuse=getter(("ECHO", t, "-6dB", f)))


def S_enc(enc, f, types):
    v = []
    for t in types:
        r = RES.get((enc, t, "-6dB", f))
        if r is None:
            return None
        v.append((r["A"]["avg"] + r["B+10"]["avg"]) / 2)
    return round(float(np.mean(v)), 4)


def S(f, types):
    a, b = S_enc("BEATs", f, types), S_enc("ECHO", f, types)
    return None if a is None or b is None else round((a + b) / 2, 4)


n2n_ok = all(SUMMARY[t]["case"] in ("I", "S") and SUMMARY[t]["n2n"].get("status") == "done" and all(SUMMARY[t]["n2n"]["G3"].values()) for t in MAIN)
cands = [f for f in CANDIDATES if (f != "N2N" or n2n_ok) and S(f, MAIN) is not None]
SC = {f: {"dev": S(f, MAIN), "conf": S(f, AUX), "dev_BEATs": S_enc("BEATs", f, MAIN), "dev_ECHO": S_enc("ECHO", f, MAIN)} for f in ["DS"] + CANDIDATES}
best = max(cands, key=lambda f: (SC[f]["dev"], -CANDIDATES.index(f)))
F1 = SC[best]["dev"] >= SC["DS"]["dev"] + BAR_F1
F2 = SC[best]["conf"] is not None and SC["DS"]["conf"] is not None and SC[best]["conf"] >= SC["DS"]["conf"] - BAR_F2
PROGRAM_FE = best if (F1 and F2) else "DS"
DEC = {"F1": {"pass": bool(F1), "best": best, "S_best": SC[best]["dev"], "S_DS": SC["DS"]["dev"], "bar": round(SC["DS"]["dev"] + BAR_F1, 4)},
       "F2": {"pass": bool(F2), "S_best": SC[best]["conf"], "S_DS": SC["DS"]["conf"],
              "bar": None if SC["DS"]["conf"] is None else round(SC["DS"]["conf"] - BAR_F2, 4)},
       "program_front_end": PROGRAM_FE, "N2N_eligible": bool(n2n_ok)}

print(f"{'type':7s} {'level':5s} {'enc':6s} {'front':7s} " + " ".join(f"{s:>7s}" for s in SPLITS))
for (enc, t, lv, f), r in sorted(RES.items(), key=lambda kv: (TYPES.index(kv[0][1]), LEVELS.index(kv[0][2]), kv[0][0], kv[0][3])):
    print(f"{t:7s} {lv:5s} {enc:6s} {f:7s} " + " ".join(f"{r[s]['avg']:7.4f}" for s in SPLITS))
print("\nscore S = mean over machines of (A + B+10)/2 at −6 dB, averaged over BEATs and ECHO:")
for f, v in SC.items():
    print(f"  {f:5s} development {v['dev']} (BEATs {v['dev_BEATs']}, ECHO {v['dev_ECHO']}) · confirmation {v['conf']}")
print(f"\nF1 · best twin front end {best}: {SC[best]['dev']} against DS {SC['DS']['dev']} + {BAR_F1} →", "pass" if F1 else "fail")
print(f"F2 · on valves and slide rails: {SC[best]['conf']} against DS {SC['DS']['conf']} − {BAR_F2} →", "pass" if F2 else "fail")
print("→ the program's front end from here on:", PROGRAM_FE)

REP = {"R_share": {}, "cause_n12": {}, "calibration_size": {}, "published_m6": {}, "meter_m6": {}, "meter_0": {}, "meter_curve_m6": {}}
for t in TYPES:
    a6 = RES[("BEATs", t, "6dB", "CH0")]["A"]["avg"]; am6 = RES[("BEATs", t, "-6dB", "CH0")]["A"]["avg"]
    REP["R_share"][t] = {f: round((RES[("BEATs", t, "-6dB", f)]["A"]["avg"] - am6) / (a6 - am6), 3) if a6 - am6 > 0.02 and ("BEATs", t, "-6dB", f) in RES else None
                         for f in FE_ALL}
    REP["published_m6"][t] = {"published": PUBLISHED_M6[t], **{f"{enc}|{PROGRAM_FE}": RES[(enc, t, "-6dB", PROGRAM_FE)]["A"]["avg"] for enc in ("BEATs", "ECHO")
                                                               if (enc, t, "-6dB", PROGRAM_FE) in RES},
                              "fused": FUSED[(t, PROGRAM_FE)]["A"]["avg"] if (t, PROGRAM_FE) in FUSED else None}
    for lv, keyname in (("-6dB", "meter_m6"), ("0dB", "meter_0")):
        if SUMMARY[t]["case"] in ("I", "S"):
            cal_t, met_t, _ = load_stats(t)
            REP[keyname][t] = {f: round(float(np.median([met_t[p][f"{f}|{lv}"]["snr"] for p in met_t])), 2) for f in METER_FE}
            if lv == "-6dB":                                   # per bin, power averaged over the type's machines (as in the picture, unsmoothed)
                REP["meter_curve_m6"][t] = {f: np.round(10 * np.log10(np.maximum(np.mean([met_t[p][f"{f}|-6dB"]["ps"] for p in met_t], 0), 1e-30)
                                                                    / np.maximum(np.mean([met_t[p][f"{f}|-6dB"]["pn"] for p in met_t], 0), 1e-30)), 2).tolist()
                                            for f in METER_FE}
for m in MAIN:
    REP["cause_n12"][m] = {"MPDR (N12/N13)": MPDR[m], **{f: {s: RES[("BEATs", m, "-6dB", f)][s]["avg"] for s in ("A", "B+10")} for f in ("MVDRd", "MVDR", "DS")}}
    REP["calibration_size"][m] = {f: {s: RES[("BEATs", m, "-6dB", f)][s]["avg"] for s in ("A", "B+10")} for f in ("MVDR10", "MVDR")}
print("\nreported:", json.dumps({k: v for k, v in REP.items() if k != "meter_curve_m6"}, indent=1))

In [ ]:
# 13 · Save, two pictures, and one zip to download
import matplotlib
import matplotlib.pyplot as plt

dc = json.loads((OUT / "dcase_check.json").read_text()) if (OUT / "dcase_check.json").exists() else None
RESULT = {"rules": RULES, "program": PROGRAM, "date": time.strftime("%Y-%m-%d %H:%M"), "test_run": TEST,
          "meta": {"device": DEVICE, "gpu": GPU, "torch": torch.__version__, "checkpoint_sha256": CKPT_SHA, "code_sha256": code_sha,
                   "echo": INFO.get("ECHO"), "zips": {k: v[1:] for k, v in ZIPS.items()}, "dcase_zip": DCASE[1:]},
          "G0": G0, "G0d_pass": G0D_PASS, "types": SUMMARY, "scores": SC, "decisions": DEC, "reported": REP, "dcase_check": dc,
          "results": {"|".join(k): v for k, v in RES.items()}, "fused": {"|".join(k): v for k, v in FUSED.items()}}
(OUT / "twin_front_ends.json").write_text(json.dumps(RESULT, indent=1, default=float))

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8d8d8d"
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey="row")
for c, t in enumerate(TYPES):
    for r, s in enumerate(("A", "B+10")):
        ax = axes[r, c]
        fes = [f for f in FE_ALL if ("BEATs", t, "-6dB", f) in RES]
        x = np.arange(len(fes))
        for j, (enc, col) in enumerate((("BEATs", BLUE), ("ECHO", ORANGE))):
            v = [RES[(enc, t, "-6dB", f)][s]["avg"] if (enc, t, "-6dB", f) in RES else np.nan for f in fes]
            ax.bar(x + (j - 0.5) * 0.38, v, 0.36, color=col, label=enc if (r == 0 and c == 0) else None)
        ax.set_xticks(x); ax.set_xticklabels(fes, fontsize=9)
        ax.set_ylim(0.4, 1.0); ax.grid(axis="y", color="#e6e6e3", lw=0.8); ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_title(f"{t} · {'known machine (A)' if s == 'A' else 'new machine, 10 clips (B+10)'}", fontsize=10)
        if c == 0:
            ax.set_ylabel("average AUC at −6 dB")
fig.legend(loc="lower center", ncol=2, frameon=False)
fig.suptitle(f"Notebook 19 · front ends at −6 dB, scorer B1 · program front end: {PROGRAM_FE}", fontsize=12)
fig.tight_layout(rect=(0, 0.05, 1, 0.95))
fig.savefig(OUT / "twin_front_ends_auc.png", dpi=150); plt.show()

COL = {"CH0": GRAY, "DS": BLUE, "MVDR": ORANGE, "MWF": "#1baf7a", "MVDRd": "#e87ba4", "MVDR10": "#008300"}
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), sharey=True)
fk = np.arange(L // 2 + 1) * SR / L / 1000
for c, t in enumerate(TYPES):
    ax = axes[c]
    if SUMMARY[t]["case"] in ("I", "S"):
        _, met_t, _ = load_stats(t)
        for f in METER_FE:
            ps = np.mean([met_t[p][f"{f}|-6dB"]["ps"] for p in met_t], 0); pn = np.mean([met_t[p][f"{f}|-6dB"]["pn"] for p in met_t], 0)
            sm = lambda v: np.convolve(v, np.ones(9) / 9, mode="same")
            ax.plot(fk[1:], 10 * np.log10(np.maximum(sm(ps), 1e-30) / np.maximum(sm(pn), 1e-30))[1:], color=COL[f], lw=2 if f in ("DS", "MVDR") else 1.2,
                    ls={"MVDRd": "--", "MVDR10": ":"}.get(f, "-"), zorder={"MVDR": 6, "DS": 5, "MWF": 4, "CH0": 3}.get(f, 2),
                    label=f if c == 0 else None)
    ax.axhline(-6, color="#c3c2b7", lw=0.8, ls="--")
    ax.set_title(f"{t} · twin-measured SNR at −6 dB", fontsize=10); ax.set_xlabel("frequency (kHz)")
    ax.grid(color="#e6e6e3", lw=0.8); ax.spines[["top", "right"]].set_visible(False)
    if c == 0:
        ax.set_ylabel("SNR after the front end (dB)")
fig.legend(loc="lower center", ncol=6, frameon=False)
fig.tight_layout(rect=(0, 0.08, 1, 1))
fig.savefig(OUT / "twin_front_ends_snr.png", dpi=150); plt.show()

ZIP = OUT / "p04_n19_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ("twin_front_ends.json", "twin_front_ends_auc.png", "twin_front_ends_snr.png", "dcase_check.json"):
        if (OUT / f).exists():
            z.write(OUT / f, f)
print("saved", ZIP, f"· {(time.time() - T_START) / 60:.0f} min in this session")
if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP))

## Send back
Send the zip that downloaded (`p04_n19_results.zip`). It is also in Drive › `P04_TwinFrontEnds`. The large files (embeddings, statistics, denoisers) stay in Drive for the next notebooks.